In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import gdown

os.makedirs("/kaggle/working/data",    exist_ok=True)
os.makedirs("/kaggle/working/models",  exist_ok=True)
os.makedirs("/kaggle/working/results", exist_ok=True)

# Download CSV
print("Downloading CSV...")
gdown.download(
    "https://drive.google.com/uc?id=1m5D9X5xBNfd25kMMB9G0CBPDWcsMc0Ye",
    "/kaggle/working/data/clean_multimodal_samples.csv",
    quiet=False
)
print("✓ CSV downloaded")

Downloading...
From: https://drive.google.com/uc?id=1m5D9X5xBNfd25kMMB9G0CBPDWcsMc0Ye
To: /kaggle/working/data/clean_multimodal_samples.csv
100%|██████████| 28.2M/28.2M [00:00<00:00, 85.2MB/s]

✓ CSV downloaded


In [3]:
import zipfile, shutil

print("Downloading ZebraMap images...")
gdown.download(
    "https://drive.google.com/uc?id=1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq",
    "/kaggle/working/zebramap_final.zip",
    quiet=False
)

print("Extracting...")
os.makedirs("/kaggle/temp/images", exist_ok=True)
with zipfile.ZipFile(
        "/kaggle/working/zebramap_final.zip", 'r') as z:
    total = len(z.namelist())
    for i, member in enumerate(z.namelist()):
        z.extract(member, "/kaggle/temp/images")
        if i % 20000 == 0:
            print(f"  {i}/{total} ({i/total*100:.0f}%)")

os.remove("/kaggle/working/zebramap_final.zip")
print("✓ Images ready")

total_imgs = sum(
    len(files) for _, _, files
    in os.walk("/kaggle/temp/images"))
print(f"✓ Total images: {total_imgs}")

Downloading...
From (original): https://drive.google.com/uc?id=1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq
From (redirected): https://drive.google.com/uc?id=1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq&confirm=t&uuid=39101a87-1895-4da2-952d-f984ea98e2c6
To: /kaggle/working/zebramap_final.zip
100%|██████████| 10.5G/10.5G [02:07<00:00, 82.6MB/s]


Extracting...
  0/110262 (0%)
  20000/110262 (18%)
  40000/110262 (36%)
  60000/110262 (54%)
  80000/110262 (73%)
  100000/110262 (91%)
✓ Images ready
✓ Total images: 79478


In [4]:
import pandas as pd
import numpy as np
import ast
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available()
                      else "cpu")
print(f"✓ Device: {torch.cuda.get_device_name(0)}")

df = pd.read_csv(
    "/kaggle/working/data/clean_multimodal_samples.csv")

new_le = LabelEncoder()
df['label'] = new_le.fit_transform(df['disease_name'])

def make_symptom_text(sym_str):
    try:
        syms = sym_str if isinstance(sym_str, list) \
                       else ast.literal_eval(sym_str)
        return ' [SEP] '.join(
            [s.lower().strip() for s in syms])
    except:
        return str(sym_str)

df['symptom_text'] = df['symptoms'].apply(make_symptom_text)

# Fix image paths
def fix_path(img_val):
    try:
        imgs = eval(img_val)
        for img in imgs:
            old = img['path']
            if 'images/' in old:
                rel = old.split('images/')[-1]
                img['path'] = f"/kaggle/temp/images/{rel}"
        return imgs
    except:
        return []

print("Fixing image paths...")
df['images_fixed'] = df['images'].apply(fix_path)

# Get first valid image path
def get_valid_image(img_val):
    try:
        imgs = img_val if isinstance(img_val, list) \
                       else eval(img_val)
        for img in imgs:
            if os.path.exists(img['path']):
                return img['path']
    except:
        pass
    return None

print("Finding valid image paths...")
df['image_path'] = df['images_fixed'].apply(
    lambda x: get_valid_image(x)
    if isinstance(x, list) else None
)

df_paired = df.dropna(
    subset=['image_path']).reset_index(drop=True)

print(f"✓ Paired samples : {len(df_paired)}")

# Filter + split
label_counts = df_paired['label'].value_counts()
valid_labels = label_counts[label_counts >= 2].index
df_filtered  = df_paired[
    df_paired['label'].isin(valid_labels)
].reset_index(drop=True)

train_df, test_df = train_test_split(
    df_filtered, test_size=0.2,
    random_state=42,
    stratify=df_filtered['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

all_labels    = sorted(train_df['label'].unique())
label_remap   = {old: new for new, old in enumerate(all_labels)}
reverse_remap = {new: old for old, new in label_remap.items()}
NUM_CLASSES   = len(all_labels)

train_df['label'] = train_df['label'].map(label_remap)
test_df['label']  = test_df['label'].map(label_remap)
test_df = test_df.dropna(
    subset=['label']).reset_index(drop=True)
test_df['label'] = test_df['label'].astype(int)

print(f"✓ Data ready")
print(f"  Train   : {len(train_df)}")
print(f"  Test    : {len(test_df)}")
print(f"  Classes : {NUM_CLASSES}")

✓ Device: Tesla T4
Fixing image paths...
Finding valid image paths...
✓ Paired samples : 30817
✓ Data ready
  Train   : 24498
  Test    : 6125
  Classes : 1115


In [8]:
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from PIL import Image

tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2")
print("✓ Tokenizer loaded")

class MultimodalDataset(Dataset):
    def __init__(self, df, tokenizer,
                 max_length=128, img_size=224):
        self.df         = df.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.transform  = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                [0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row['symptom_text']),
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        try:
            img = Image.open(
                row['image_path']).convert('RGB')
            img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'image'         : img,
            'label'         : torch.tensor(
                                row['label'],
                                dtype=torch.long)
        }

# Subsample 40% for speed
train_small = (
    train_df.groupby('label', group_keys=False)
    .apply(lambda x: x.sample(frac=0.4, random_state=42)
           if len(x) > 2 else x)
    .reset_index(drop=True)
)

train_ds = MultimodalDataset(train_small, tokenizer)
test_ds  = MultimodalDataset(test_df,     tokenizer)

train_loader = DataLoader(
    train_ds, batch_size=16,
    shuffle=True,  num_workers=2)
test_loader  = DataLoader(
    test_ds,  batch_size=16,
    shuffle=False, num_workers=2)

print(f"✓ DataLoaders ready")
print(f"  Train : {len(train_ds)}")
print(f"  Test  : {len(test_ds)}")
print(f"  Batches: {len(train_loader)}")

✓ Tokenizer loaded
✓ DataLoaders ready
  Train : 10064
  Test  : 6125
  Batches: 629


/tmp/ipykernel_57/2286936795.py:54: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.4, random_state=42)


In [9]:
import torchvision.models as models
from transformers import AutoModel
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

class MultimodalFusionModel(nn.Module):
    def __init__(self, num_classes,
                 text_dim=768, img_dim=2048,
                 fusion_dim=512, dropout=0.4):
        super().__init__()
        self.bert = AutoModel.from_pretrained(
            "dmis-lab/biobert-base-cased-v1.2")
        for i, layer in enumerate(
                self.bert.encoder.layer):
            if i < 8:
                for param in layer.parameters():
                    param.requires_grad = False

        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for layer in list(backbone.children())[:-3]:
            for param in layer.parameters():
                param.requires_grad = False
        self.resnet = nn.Sequential(
            *list(backbone.children())[:-1])

        self.text_proj = nn.Linear(text_dim, fusion_dim)
        self.img_proj  = nn.Linear(img_dim,  fusion_dim)
        self.fusion    = nn.Sequential(
            nn.LayerNorm(fusion_dim * 2),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_classes)
        )

    def forward(self, input_ids, attention_mask, images):
        text_out  = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask)
        text_feat = text_out.last_hidden_state[:, 0, :]
        text_proj = self.text_proj(text_feat)
        img_feat  = self.resnet(images).squeeze(-1).squeeze(-1)
        img_proj  = self.img_proj(img_feat)
        fused     = torch.cat([text_proj, img_proj], dim=1)
        return self.fusion(fused)


fusion_model = MultimodalFusionModel(NUM_CLASSES).to(device)

trainable = sum(p.numel() for p in fusion_model.parameters()
                if p.requires_grad)
print(f"✓ Fusion model built")
print(f"  Trainable params : {trainable:,}")
print(f"  Classes          : {NUM_CLASSES}")

# Test forward pass
batch = next(iter(train_loader))
with torch.no_grad():
    out = fusion_model(
        batch['input_ids'].to(device),
        batch['attention_mask'].to(device),
        batch['image'].to(device)
    )
print(f"  Output shape     : {out.shape}")

# Train
EPOCHS    = 10
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(
    filter(lambda p: p.requires_grad,
           fusion_model.parameters()),
    lr=1e-5, weight_decay=0.01)
scheduler = CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-7)

print(f"\nTraining Fusion Model — {EPOCHS} epochs")
print("-" * 55)

for epoch in range(EPOCHS):
    fusion_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in train_loader:
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        images = batch['image'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = fusion_model(ids, mask, images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            fusion_model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    acc      = correct / total * 100
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%")

print("-" * 55)
print("✓ Training complete")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Fusion model built
  Trainable params : 69,113,691
  Classes          : 1115
  Output shape     : torch.Size([16, 1115])

Training Fusion Model — 10 epochs
-------------------------------------------------------
Epoch 01/10 | Loss: 6.7713 | Train Acc: 1.06%
Epoch 02/10 | Loss: 6.4002 | Train Acc: 1.76%
Epoch 03/10 | Loss: 6.1701 | Train Acc: 3.52%
Epoch 04/10 | Loss: 5.9990 | Train Acc: 4.38%
Epoch 05/10 | Loss: 5.8416 | Train Acc: 5.79%
Epoch 06/10 | Loss: 5.7263 | Train Acc: 6.61%
Epoch 07/10 | Loss: 5.6375 | Train Acc: 7.34%
Epoch 08/10 | Loss: 5.5718 | Train Acc: 8.57%
Epoch 09/10 | Loss: 5.5308 | Train Acc: 8.63%
Epoch 10/10 | Loss: 5.5176 | Train Acc: 8.76%
-------------------------------------------------------
✓ Training complete


In [10]:
from sklearn.metrics import accuracy_score, f1_score
import json

def evaluate_fusion_topk(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    topk_correct = {1: 0, 3: 0, 5: 0}
    total = 0

    with torch.no_grad():
        for batch in loader:
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            images = batch['image'].to(device)
            labels = batch['label'].to(device)

            logits = model(ids, mask, images)
            preds  = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for k_val in [1, 3, 5]:
                k_act = min(k_val, logits.size(1))
                topk  = logits.topk(k_act, dim=1).indices
                for i, lbl in enumerate(labels):
                    if lbl in topk[i]:
                        topk_correct[k_val] += 1
            total += labels.size(0)

    return {
        "accuracy"     : round(accuracy_score(
                            all_labels, all_preds)*100, 2),
        "f1_macro"     : round(f1_score(
                            all_labels, all_preds,
                            average='macro',
                            zero_division=0)*100, 2),
        "top1_accuracy": round(topk_correct[1]/total*100, 2),
        "top3_accuracy": round(topk_correct[3]/total*100, 2),
        "top5_accuracy": round(topk_correct[5]/total*100, 2),
        "total_samples": total
    }

print("Evaluating...")
fusion_metrics = evaluate_fusion_topk(
    fusion_model, test_loader, device)

print("\n" + "=" * 55)
print("FUSION MODEL — TOP-K RESULTS")
print("=" * 55)
print(f"  Accuracy     : {fusion_metrics['accuracy']}%")
print(f"  F1 Macro     : {fusion_metrics['f1_macro']}%")
print(f"  Top-1 Acc    : {fusion_metrics['top1_accuracy']}%")
print(f"  Top-3 Acc    : {fusion_metrics['top3_accuracy']}%")
print(f"  Top-5 Acc    : {fusion_metrics['top5_accuracy']}%")
print(f"  Test samples : {fusion_metrics['total_samples']}")

# Save model
import pickle
torch.save({
    'model_state_dict': fusion_model.state_dict(),
    'label_remap'     : label_remap,
    'reverse_remap'   : reverse_remap,
    'num_classes'     : NUM_CLASSES,
    'metrics'         : fusion_metrics
}, "/kaggle/working/models/fusion_model.pt")

# Save label encoder
with open("/kaggle/working/models/label_encoder.pkl", "wb") as f:
    pickle.dump(new_le, f)

# Save summary
summary = {
    "day"        : "14-15-16",
    "model"      : "BioBERT + ResNet-50 Late Fusion",
    "num_classes": NUM_CLASSES,
    "train_size" : len(train_ds),
    "test_size"  : len(test_ds),
    "epochs"     : EPOCHS,
    "metrics"    : fusion_metrics,
    "status"     : "Day 14-16 complete"
}
with open("/kaggle/working/results/day14_16_summary.json","w") as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Fusion model saved")
print(f"✓ Label encoder saved")
print(f"✓ Summary saved")
print(f"\n{'='*55}")
print(f"DAY 14-15-16 COMPLETE ✓")
print(f"{'='*55}")
print(f"  Top-1 : {fusion_metrics['top1_accuracy']}%")
print(f"  Top-3 : {fusion_metrics['top3_accuracy']}%")
print(f"  Top-5 : {fusion_metrics['top5_accuracy']}%")
print(f"\n  Next → Day 17: FastAPI Backend")

Evaluating...

FUSION MODEL — TOP-K RESULTS
  Accuracy     : 8.52%
  F1 Macro     : 0.58%
  Top-1 Acc    : 8.52%
  Top-3 Acc    : 16.95%
  Top-5 Acc    : 21.22%
  Test samples : 6125

✓ Fusion model saved
✓ Label encoder saved
✓ Summary saved

DAY 14-15-16 COMPLETE ✓
  Top-1 : 8.52%
  Top-3 : 16.95%
  Top-5 : 21.22%

  Next → Day 17: FastAPI Backend
